## 🔁 LangGraph Lab
Explore stateful agent orchestration with LangGraph to manage dynamic flows in AI systems.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated
import operator

# Define state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# Calculator tool
def calculate(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

# Setup LLM
llm = ChatOllama(model="mixtral")

# Custom prompt that instructs the LLM to call the calculator
tool_prompt = PromptTemplate.from_template(
    "You are an assistant. If the user input is a math expression, say 'TOOL: {expression}'\nUser: {query}"
)

# Agent node
def agent(state: AgentState):
    user_msg = state["messages"][-1]["content"]
    prompt = tool_prompt.format(query=user_msg, expression=user_msg)
    response = llm.invoke(prompt)
    return {"messages": [{"role": "assistant", "content": response}]}

# Tool node
def call_tool(state: AgentState):
    last_msg = state["messages"][-1]["content"]
    if last_msg.startswith("TOOL:"):
        expression = last_msg.replace("TOOL:", "").strip()
        result = calculate(expression)
        return {"messages": [{"role": "tool", "content": result}]}
    return {"messages": []}

# Create graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent)
graph.add_node("tool", call_tool)
graph.add_edge("agent", "tool")
graph.add_conditional_edges("tool", lambda state: "agent" if state["messages"] else END)
graph.set_entry_point("agent")

# Compile app
app = graph.compile()

# Run queries
queries = ["What is 5 * 3?", "What is LangGraph?"]
for query in queries:
    result = app.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"Query: {query}\nAnswer: {result['messages'][-1]['content']}\n")


In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.llms import Ollama
from typing import TypedDict, Annotated
import operator

# Define state
class SupportState(TypedDict):
    messages: Annotated[list, operator.add]

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent node
def agent(state: SupportState):
    messages = state["messages"]
    prompt = f"You are a customer support agent. Use the conversation history to answer:\n{messages}"
    response = llm.invoke(prompt)
    return {"messages": [{"content": response, "role": "assistant"}]}

# Define graph
graph = StateGraph(SupportState)
graph.add_node("agent", agent)
graph.add_edge("agent", END)
graph.set_entry_point("agent")

# Compile graph
app = graph.compile()

# Run conversation
queries = [
    "What is the status of Alice's order?",
    "Can she get a refund?"
]
state = {"messages": []}
for query in queries:
    state = app.invoke({"messages": [{"content": query, "role": "user"}] + state["messages"]})
    print(f"Query: {query}\nAnswer: {state['messages'][-1]['content']}\n")